If your body is a system, protein kinases behave like tiny machines. There's a type of protein EGFR (Epidermal Growth Factor) that acts like a light switch that causes cells to grow; sometimes the switch gets stuck and leads to cancer.


When modeling the X input will be the EGFR features (phospho, protein, RNA, activity) but the mutation type will influence this.

In [3]:
#pip install cptac

In [2]:
import pandas as pd
import cptac

In [43]:
mutation = pd.read_csv('data/egfr_mutation.csv')
"""
removed WT; no EGFR mutation present
if needed it's mutations_raw.csv

EFGR = mutation
"""
mutation = mutation.rename(columns={"EGFR": "mutation"})
#renamed following to match with other cptac protein/rna df
mutation = mutation.rename(columns={"Patient_ID": "PATIENT_ID"})
mutation["PATIENT_ID"] = mutation["PATIENT_ID"].str.replace(r"-T.*", "", regex=True)
mutation.head()

"""
#find a way to remove for loop (Markus dock points)
def classify_mutation(mutation):
    if pd.isna(mutation):
        return "Unknown"
    m = mutation.lower()
    if "l858r" in m:
        return "L858R"
    elif "del" in m or "e746" in m or "exon19" in m:
        return "Exon19"
    else:
        return "Other"
"""

def classify_column(df):
    m = df['mutation'].str.lower()
    
    df['classification'] = "Other"
    # mutation points for exon 19 and L858R
    df.loc[m.str.contains("del|e746|exon19", na=False), 'classification'] = "Exon19"
    df.loc[m.str.contains("l858r", na=False), 'classification'] = "L858R"
    df.loc[df['mutation'].isna(), 'classification'] = "Unknown"
    
    return df

egfr_group_counts = mutation['mutation'].value_counts()
print("Frequency of each EGFR group type:")
#display(egfr_group_counts)
mutation.to_csv("data/egfr_mutation.csv", index=False)

mutation = classify_column(mutation)
mutation.head()

Frequency of each EGFR group type:


,PATIENT_ID,mutation,EGFR_type,classification
0,TCGA-05-4382-01,R222L E545Q,Other,Other
1,TCGA-05-4402-01,T751_I759delinsN I759N,Exon19,Exon19
2,TCGA-05-4410-01,R377S,Other,Other
3,TCGA-05-5423-01,L833F L861Q,Other,Other
4,TCGA-17-Z026-01,G721V,Other,Other


In [6]:
luad_data = cptac.Luad()
proteomics = luad_data.get_proteomics(source='bcm')
phospho = luad_data.get_phosphoproteomics(source='bcm')
rna_seq = luad_data.get_transcriptomics(source='bcm')
#somatic_mutations = luad_data.get_somatic_mutation()

egfr_rna = rna_seq[["EGFR"]]
egfr_proteomics = proteomics[["EGFR"]]

Why do we need all four types of CPTAC data? 

Phospho 
- capture activation state (site-specfic)
   - Y1068
   - Y1173
   - Y992

Protein
- total EGFR abundance 
- baseline feature

rNA
- expression level

Activity
- Functional output


In [7]:
luad_data = cptac.Luad()
"""
Only need to be downloaded once 

download:
log2 ref normalized proteomic
log2 red tumor/normal proteomic
log2 ref normalized phosphoproteomic
log2 red tumor/normal phosphoproteomic
log2 ref normalized transcriptomic
log2 red tumor/normal transcriptomic


rows are patients and columns are genes (not an index = need to change)
"""
proteomics = luad_data.get_proteomics(source='bcm')
phospho = luad_data.get_phosphoproteomics(source='bcm')
rna_seq = luad_data.get_transcriptomics(source='bcm')

#extract egfr data for kinase activity
egfr_rna = rna_seq[["EGFR"]]
egfr_proteomics = proteomics[["EGFR"]]

egfr_phospho = phospho[["EGFR"]]
#site_level_phospho = egfr_phospho.filter(regex='Y1068|Y1173')
site_level = egfr_phospho.columns.get_level_values("Site")
target_sites = ['Y1068', 'Y1173'] #binding site
select_cols = [col for col in egfr_phospho.columns if any(site in col for site in target_sites)]

#mean from phoso data
egfr_activity = egfr_phospho[select_cols]
egfr_activity["EGFR_activity_mean"] = egfr_activity.mean(axis=1)

#how to print rna/protein/phospho data for EGFR
print("\nEGFR RNA expression:")
display(egfr_rna)

print("\nEGFR protein expression:")
display(egfr_proteomics)

print("\nEGFR phospho expression:")
display(egfr_phospho)

"""
egfr_rna.to_csv("egfr_rna.csv")
egfr_proteomics.to_csv("egfr_protein.csv")
egfr_phospho.to_csv("egfr_phospho_all.csv") #need to organize!!
egfr_activity.to_csv("egfr_activity.csv")
"""


EGFR RNA expression:


C:\Users\anaso\AppData\Local\Temp\ipykernel_2892\353710443.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  egfr_activity["EGFR_activity_mean"] = egfr_activity.mean(axis=1)


Name,EGFR
Database_ID,ENSG00000146648.18
Patient_ID,
11LU013,16.96
11LU016,14.41
11LU022,14.42
11LU035,13.12
C3L-00001,17.07
...,...
C3N-02582.N,12.55
C3N-02586.N,12.34



EGFR protein expression:


Name,EGFR
Database_ID,ENSG00000146648.18
Patient_ID,
C3L-00001,28.192875
C3L-00009,25.219585
C3L-00080,25.238803
C3L-00083,25.041583
C3L-00093,24.469511
...,...
C3N-02582.N,25.652043
C3N-02586.N,25.306163



EGFR phospho expression:


Name                      EGFR                                        \
Site                      S991              S1064               T693   
Peptide        DERMHLPSPTDSNFY    SCPIKEDSFLQRYSS    RELVEPLTPSGEAPN   
Database_ID ENSG00000146648.18 ENSG00000146648.18 ENSG00000146648.18   
Patient_ID                                                             
C3L-00001            23.423220          26.716883          25.975380   
C3L-00009            22.750016          22.558917          24.001826   
C3L-00080            21.850419          21.632361          23.185327   
C3L-00083            21.916126                NaN          22.915656   
C3L-00093            22.427414          20.986129          23.373010   
...                        ...                ...                ...   
C3N-02582.N                NaN                NaN                NaN   
C3N-02586.N                NaN                NaN                NaN   
C3N-02587.N                NaN                NaN                NaN   
C3N-02588.N                NaN                NaN                NaN   
C3N-02729.N                NaN                NaN                NaN   

Name                                                                  \
Site                     S1039              Y1197              S1071   
Peptide        TPLLSSLSATSNNST    STAENAEYLRVAPQS    SFLQRYSSDPTGALT   
Database_ID ENSG00000146648.18 ENSG00000146648.18 ENSG00000146648.18   
Patient_ID                                                             
C3L-00001            25.576142          24.880159                NaN   
C3L-00009            22.688238          20.389850          18.333627   
C3L-00080            23.004742          19.718586                NaN   
C3L-00083            22.892887          20.152819                NaN   
C3L-00093            22.675998          20.677698          18.467871   
...                        ...                ...                ...   
C3N-02582.N                NaN                NaN                NaN   
C3N-02586.N                NaN                NaN                NaN   
C3N-02587.N                NaN                NaN                NaN   
C3N-02588.N                NaN                NaN                NaN   
C3N-02729.N                NaN                NaN                NaN   

Name                                                                  \
Site                     S1042              S1166              Y1172   
Peptide        LSSLSATSNNSTVAC    QKGSHQISLDNPDYQ    ISLDNPDYQQDFFPK   
Database_ID ENSG00000146648.18 ENSG00000146648.18 ENSG00000146648.18   
Patient_ID                                                             
C3L-00001            21.246517          26.654267          21.924476   
C3L-00009            18.006135          22.035377          18.590884   
C3L-00080                  NaN                NaN          17.556575   
C3L-00083                  NaN                NaN                NaN   
C3L-00093            17.877844                NaN                NaN   
...                        ...                ...                ...   
C3N-02582.N                NaN                NaN                NaN   
C3N-02586.N                NaN                NaN                NaN   
C3N-02587.N                NaN                NaN                NaN   
C3N-02588.N                NaN                NaN                NaN   
C3N-02729.N                NaN                NaN                NaN   

Name                            ...                                        \
Site                     S1026  ...              T1041               S695   
Peptide        PQQGFFSSPSTSRTP  ...    LLSSLSATSNNSTVA    LVEPLTPSGEAPNQA   
Database_ID ENSG00000146648.18  ... ENSG00000146648.18 ENSG00000146648.18   
Patient_ID                      ...                                         
C3L-00001                  NaN  ...                NaN                NaN   
C3L-00009                  NaN  ...                NaN                NaN   
C3

'\negfr_rna.to_csv("egfr_rna.csv")\negfr_proteomics.to_csv("egfr_protein.csv")\negfr_phospho.to_csv("egfr_phospho_all.csv") #need to organize!!\negfr_activity.to_csv("egfr_activity.csv")\n'

In [ ]:
#organizing egfr_phospho_all dataset from multi 
phospho_raw = pd.read_csv("data/egfr_phospho_all.csv", header=None)
gene_row = phospho_raw.iloc[0]
site_row = phospho_raw.iloc[1]
peptide_row = phospho_raw.iloc[2]
columns = []
for i in range(len(site_row)):
    site = str(site_row[i])
    peptide = str(peptide_row[i])
    columns.append(f"{site}_{peptide}")

df_clean = phospho_raw.iloc[5:].copy()
df_clean.columns = ["patient_id"] + columns[1:]
df_clean = df_clean.reset_index(drop=True)

df_clean['patient_id'] = df_clean['patient_id'].str.strip().str.upper()
df_clean = df_clean.rename(columns={'patient_id': 'PATIENT_ID'})

df_clean.head()

#df_clean.to_csv("data/egfr_phospho_cleaned.csv", index=False)
#egfr_phospho_cleaned = pd.read_csv("data/egfr_phospho_cleaned.csv")
#egfr_phospho_cleaned.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 207 entries, 0 to 206
Data columns (total 43 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   PATIENT_ID               207 non-null    object 
 1   S991_DERMHLPSPTDSNFY     106 non-null    float64
 2   S1064_SCPIKEDSFLQRYSS    101 non-null    float64
 3   T693_RELVEPLTPSGEAPN     101 non-null    float64
 4   S1039_TPLLSSLSATSNNST    92 non-null     float64
 5   Y1197_STAENAEYLRVAPQS    90 non-null     float64
 6   S1071_SFLQRYSSDPTGALT    45 non-null     float64
 7   S1042_LSSLSATSNNSTVAC    44 non-null     float64
 8   S1166_QKGSHQISLDNPDYQ    44 non-null     float64
 9   Y1172_ISLDNPDYQQDFFPK    44 non-null     float64
 10  S1026_PQQGFFSSPSTSRTP    41 non-null     float64
 11  Y1092_TFLPVPEYINQSVPK    34 non-null     float64
 12  T993_RMHLPSPTDSNFYRA     18 non-null     float64
 13  S695_LVEPLTPSGEAPNQA     16 non-null     float64
 14  T1041_LLSSLSATSNNSTVA    1

In [30]:
data = pd.read_csv("data/egfr_phospho_cleaned.csv")
data.head()

,PATIENT_ID,S991_DERMHLPSPTDSNFY,S1064_SCPIKEDSFLQRYSS,T693_RELVEPLTPSGEAPN,S1039_TPLLSSLSATSNNST,Y1197_STAENAEYLRVAPQS,S1071_SFLQRYSSDPTGALT,S1042_LSSLSATSNNSTVAC,S1166_QKGSHQISLDNPDYQ,Y1172_ISLDNPDYQQDFFPK,...,T1041_LLSSLSATSNNSTVA.1,S695_LVEPLTPSGEAPNQA.1,T993_RMHLPSPTDSNFYRA.1,S1045_LSATSNNSTVACIDR.1,S1025_IPQQGFFSSPSTSRT.1,S1081_TGALTEDSIDDTFLP.1,T1085_TEDSIDDTFLPVPEY.1,Y1016_DVVDADEYLIPQQGF.1,Y1069_EDSFLQRYSSDPTGA.1,Y1110_GSVQNPVYHNQPLNP.1
0,C3L-00001,23.423220,26.716883,25.975380,25.576142,24.880159,NaN,21.246517,26.654267,21.924476,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C3L-00009,22.750016,22.558917,24.001826,22.688238,20.389850,18.333627,18.006135,22.035377,18.590884,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,C3L-00080,21.850419,21.632361,23.185327,23.004742,19.718586,NaN,NaN,NaN,17.556575,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,C3L-00083,21.916126,NaN,22.915656,22.892887,20.152819,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,C3L-00093,22.427414,20.986129,23.373010,22.675998,20.677698,18.467871,17.877844,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
#printed egfr_phospho_cleaned.info() Y1172_ISLDNPDYQQDFFPK column
print(egfr_phospho_cleaned['Y1172_ISLDNPDYQQDFFPK'].head())


0    21.924476
1    18.590884
2    17.556575
3          NaN
4          NaN
Name: Y1172_ISLDNPDYQQDFFPK, dtype: float64


In [40]:
df = pd.read_csv('data/egfr_phospho_cleaned.csv')
target_sites = ['Y1172_ISLDNPDYQQDFFPK', 'Y1173_ISLDNPDYQQDFFPK', 'Y1069_EDSFLQRYSSDPTGA', 'Y1068_EDSFLQRYSSDPTGA']

cols_to_keep = ['PATIENT_ID'] + [col for col in df.columns if any(site in col for site in target_sites)]
df_filtered = df[cols_to_keep]
df_filtered.head()
df_filtered.to_csv("data/egfr_phospho_tcga.csv", index=False)

In [35]:
df = pd.read_csv('data/egfr_phospho_cleaned.csv')
target_sites = ['Y1172_ISLDNPDYQQDFFPK', 'Y1173_ISLDNPDYQQDFFPK']

cols_to_keep = ['PATIENT_ID'] + [col for col in df.columns if any(site in col for site in target_sites)]
df_filtered = df[cols_to_keep].copy()
df_filtered['PATIENT_ID'] = df_filtered['PATIENT_ID'].str.split('.').str[0]
df_filtered['PATIENT_ID'] = df_filtered['PATIENT_ID'].str.replace(r'-T.*', '', regex=True)
tcga_df = df_filtered[df_filtered['PATIENT_ID'].str.startswith('TCGA')].copy()

tcga_df.to_csv('data/egfr_phospho_tcga.csv', index=False)

print(f"Columns kept: {df_filtered.columns.tolist()}")
print(f"Number of TCGA rows: {len(tcga_df)}")

Columns kept: ['PATIENT_ID', 'Y1172_ISLDNPDYQQDFFPK', 'Y1172_ISLDNPDYQQDFFPK.1']
Number of TCGA rows: 0


If the mutation.csv can show if the EGFR was mutated through L858R or exon 19 it can be indicator if the EGFR-mutated tumors show higher phospho activity?

Need to csv according to EGFR with CPTAC before modeling Binding Affinity

In [ ]:
#do they match? sanity check


In [ ]:
mut_egfr = pd.read_csv('data/mutation.csv')
mut_egfr = mut_egfr.rename(columns={"SAMPLE_ID": "Patient_ID"})
mut_egfr["Patient_ID"] = mut_egfr["Patient_ID"].str.replace(r"-T.*", "", regex=True)

#flatten to merge
egfr_activity = rna_seq[["EGFR"]].copy()
egfr_activity.columns = egfr_activity.columns.get_level_values(0)
egfr_activity = egfr_activity.reset_index() #patient id = index
merged_df = mutation.merge(egfr_activity, on="Patient_ID")

#do they match? sanity check
print("csv patient index", mutation["Patient_ID"].head().tolist())
print("EGFR activity patient index", egfr_activity["Patient_ID"].head().tolist())
egfr_activity["Patient_ID"] = egfr_activity["Patient_ID"].str.replace(r"\.N$", "", regex=True)
merged_df = mutation.merge(egfr_activity, on="Patient_ID")
print(f"Merged rows: {len(merged_df)}")



KeyError: 'Gene'